In [1]:
# ============================================================
# CLASSIFIER STUDY — STEP 1
# Load Geneformer embeddings
# ============================================================

import numpy as np
import pandas as pd
import os

BASE_DIR = r"F:\GeneformerProject\data\geneformer_embeddings"


# ------------------------------------------------------------
# Load embeddings
# ------------------------------------------------------------

X_train = np.load(os.path.join(BASE_DIR, "train_embeddings.npy"))
X_val   = np.load(os.path.join(BASE_DIR, "validation_embeddings.npy"))
X_test  = np.load(os.path.join(BASE_DIR, "test_embeddings.npy"))

# ------------------------------------------------------------
# Load labels
# ------------------------------------------------------------

y_train = np.load(os.path.join(BASE_DIR, "train_labels.npy"))
y_val   = np.load(os.path.join(BASE_DIR, "validation_labels.npy"))
y_test  = np.load(os.path.join(BASE_DIR, "test_labels.npy"))

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nNumber of classes:", len(np.unique(y_train)))

Train: (7742, 768) (7742,)
Validation: (1659, 768) (1659,)
Test: (1659, 768) (1659,)

Number of classes: 33


In [2]:
# ============================================================
# COMMON EVALUATION FUNCTION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

def evaluate_classifier(model_name, y_true, y_pred):

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(
            y_true,
            y_pred,
            average="macro"
        ),
        "Weighted_F1": f1_score(
            y_true,
            y_pred,
            average="weighted"
        ),
        "Macro_Precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    }

    return results

In [3]:
# ============================================================
# 1. LOGISTIC REGRESSION
# ============================================================

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    C=1.0,
    max_iter=2000,
    solver="lbfgs",
    random_state=42
)

lr.fit(X_train, y_train)

lr_val_pred = lr.predict(X_val)
lr_test_pred = lr.predict(X_test)

lr_val_results = evaluate_classifier(
    "Logistic Regression",
    y_val,
    lr_val_pred
)

lr_test_results = evaluate_classifier(
    "Logistic Regression",
    y_test,
    lr_test_pred
)

print("Validation:")
print(lr_val_results)

print("\nTest:")
print(lr_test_results)

Validation:
{'Model': 'Logistic Regression', 'Accuracy': 0.9234478601567209, 'Macro_F1': 0.8836949038115063, 'Weighted_F1': 0.9205061551456797, 'Macro_Precision': 0.8881406387320554, 'Macro_Recall': 0.8845673371949735}

Test:
{'Model': 'Logistic Regression', 'Accuracy': 0.9216395418927065, 'Macro_F1': 0.8922711984052399, 'Weighted_F1': 0.9198155711937466, 'Macro_Precision': 0.8995476773949193, 'Macro_Recall': 0.8890699393445224}


In [4]:
# ============================================================
# 2. LINEAR NEURAL HEAD
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

# ------------------------------------------------------------
# Convert to tensors
# ------------------------------------------------------------

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)


# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

class LinearHead(nn.Module):

    def __init__(self, input_dim=768, num_classes=33):
        super().__init__()

        self.classifier = nn.Linear(
            input_dim,
            num_classes
        )

    def forward(self, x):
        return self.classifier(x)


linear_head = LinearHead(
    input_dim=768,
    num_classes=33
).to(device)

print(linear_head)

# ============================================================
# TRAIN LINEAR HEAD
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    linear_head.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

num_epochs = 30

for epoch in range(num_epochs):

    linear_head.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = linear_head(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # -----------------------------
    # Validation
    # -----------------------------

    linear_head.eval()

    with torch.no_grad():

        val_logits = linear_head(
            X_val_tensor.to(device)
        )

        val_pred = torch.argmax(
            val_logits,
            dim=1
        ).cpu().numpy()

    val_f1 = f1_score(
        y_val,
        val_pred,
        average="macro"
    )

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

 # ============================================================
# EVALUATE LINEAR HEAD
# ============================================================

linear_head.eval()

with torch.no_grad():

    test_logits = linear_head(
        X_test_tensor.to(device)
    )

    test_pred = torch.argmax(
        test_logits,
        dim=1
    ).cpu().numpy()

linear_test_results = evaluate_classifier(
    "Linear Neural Head",
    y_test,
    test_pred
)

print("\nLinear Neural Head — Test")
print("--------------------------------")

for key, value in linear_test_results.items():

    if key != "Model":
        print(f"{key}: {value:.4f}")

Device: cuda
LinearHead(
  (classifier): Linear(in_features=768, out_features=33, bias=True)
)
Epoch 01/30 | Loss: 2.4709 | Val Macro-F1: 0.4034
Epoch 02/30 | Loss: 1.4812 | Val Macro-F1: 0.6254
Epoch 03/30 | Loss: 1.0957 | Val Macro-F1: 0.7215
Epoch 04/30 | Loss: 0.8865 | Val Macro-F1: 0.7554
Epoch 05/30 | Loss: 0.7729 | Val Macro-F1: 0.7741
Epoch 06/30 | Loss: 0.6797 | Val Macro-F1: 0.7781
Epoch 07/30 | Loss: 0.6264 | Val Macro-F1: 0.7977
Epoch 08/30 | Loss: 0.5692 | Val Macro-F1: 0.7952
Epoch 09/30 | Loss: 0.5337 | Val Macro-F1: 0.7912
Epoch 10/30 | Loss: 0.5090 | Val Macro-F1: 0.8262
Epoch 11/30 | Loss: 0.4742 | Val Macro-F1: 0.8082
Epoch 12/30 | Loss: 0.4576 | Val Macro-F1: 0.8112
Epoch 13/30 | Loss: 0.4377 | Val Macro-F1: 0.8317
Epoch 14/30 | Loss: 0.4147 | Val Macro-F1: 0.8429
Epoch 15/30 | Loss: 0.3987 | Val Macro-F1: 0.8502
Epoch 16/30 | Loss: 0.3866 | Val Macro-F1: 0.8349
Epoch 17/30 | Loss: 0.3750 | Val Macro-F1: 0.8376
Epoch 18/30 | Loss: 0.3631 | Val Macro-F1: 0.8453
Epoch

In [5]:
# ============================================================
# 3. MLP CLASSIFIER
# ============================================================

class MLPClassifier(nn.Module):

    def __init__(
        self,
        input_dim=768,
        hidden_dim=256,
        num_classes=33,
        dropout=0.30
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                num_classes
            )
        )

    def forward(self, x):

        return self.network(x)


mlp = MLPClassifier(
    input_dim=768,
    hidden_dim=256,
    num_classes=33,
    dropout=0.30
).to(device)

print(mlp)
# ============================================================
# TRAIN MLP
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    mlp.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


num_epochs = 200

best_val_f1 = 0.0
best_state = None

for epoch in range(num_epochs):

    mlp.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = mlp(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    # -----------------------------
    # Validation
    # -----------------------------

    mlp.eval()

    with torch.no_grad():

        val_logits = mlp(
            X_val_tensor.to(device)
        )

        val_pred = torch.argmax(
            val_logits,
            dim=1
        ).cpu().numpy()

    val_f1 = f1_score(
        y_val,
        val_pred,
        average="macro"
    )

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1

        best_state = {
            k: v.cpu().clone()
            for k, v in mlp.state_dict().items()
        }

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Loss: {total_loss/len(train_loader):.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

# Restore best validation model
mlp.load_state_dict(best_state)

print("\nBest Validation Macro-F1:", best_val_f1)

# ============================================================
# EVALUATE MLP
# ============================================================

mlp.eval()

with torch.no_grad():

    test_logits = mlp(
        X_test_tensor.to(device)
    )

    test_pred = torch.argmax(
        test_logits,
        dim=1
    ).cpu().numpy()

mlp_test_results = evaluate_classifier(
    "MLP",
    y_test,
    test_pred
)

print("\nMLP — Test")
print("----------------")

for key, value in mlp_test_results.items():

    if key != "Model":
        print(f"{key}: {value:.4f}")


MLPClassifier(
  (network): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=33, bias=True)
  )
)
Epoch 01/200 | Loss: 2.6621 | Val Macro-F1: 0.2817
Epoch 02/200 | Loss: 1.6079 | Val Macro-F1: 0.4868
Epoch 03/200 | Loss: 1.1456 | Val Macro-F1: 0.6417
Epoch 04/200 | Loss: 0.9184 | Val Macro-F1: 0.6984
Epoch 05/200 | Loss: 0.7939 | Val Macro-F1: 0.6973
Epoch 06/200 | Loss: 0.7054 | Val Macro-F1: 0.7563
Epoch 07/200 | Loss: 0.6497 | Val Macro-F1: 0.7354
Epoch 08/200 | Loss: 0.6024 | Val Macro-F1: 0.8142
Epoch 09/200 | Loss: 0.5690 | Val Macro-F1: 0.7970
Epoch 10/200 | Loss: 0.5237 | Val Macro-F1: 0.8037
Epoch 11/200 | Loss: 0.5089 | Val Macro-F1: 0.8044
Epoch 12/200 | Loss: 0.4811 | Val Macro-F1: 0.8216
Epoch 13/200 | Loss: 0.4671 | Val Macro-F1: 0.8178
Epoch 14/200 | Loss: 0.4589 | Val Macro-F1: 0.8297
Epoch 15/200 | Loss: 0.4444 | Val Macro-F1: 0.8414
Epoch 16/20

In [6]:
# ============================================================
# 4. LINEAR SVM
# ============================================================

from sklearn.svm import LinearSVC

svm = LinearSVC(
    C=1.0,
    max_iter=5000,
    random_state=42
)

svm.fit(
    X_train,
    y_train
)

svm_val_pred = svm.predict(X_val)
svm_test_pred = svm.predict(X_test)

svm_val_results = evaluate_classifier(
    "Linear SVM",
    y_val,
    svm_val_pred
)

svm_test_results = evaluate_classifier(
    "Linear SVM",
    y_test,
    svm_test_pred
)

print("Validation:")
print(svm_val_results)

print("\nTest:")
print(svm_test_results)

Validation:
{'Model': 'Linear SVM', 'Accuracy': 0.9246534056660639, 'Macro_F1': 0.8860263493332667, 'Weighted_F1': 0.922793490059326, 'Macro_Precision': 0.895489353731008, 'Macro_Recall': 0.8848248850959465}

Test:
{'Model': 'Linear SVM', 'Accuracy': 0.9204339963833634, 'Macro_F1': 0.8962956303356279, 'Weighted_F1': 0.9193827751200303, 'Macro_Precision': 0.9021037139082423, 'Macro_Recall': 0.893106289600127}


In [7]:
# ============================================================
# 5. XGBOOST
# ============================================================

from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=33,

    n_estimators=500,
    max_depth=6,
    learning_rate=0.001,

    subsample=0.7,
    colsample_bytree=0.8,

    reg_lambda=1.0,

    eval_metric="mlogloss",

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)

xgb.fit(
    X_train,
    y_train
)

xgb_val_pred = xgb.predict(X_val)
xgb_test_pred = xgb.predict(X_test)

xgb_val_results = evaluate_classifier(
    "XGBoost",
    y_val,
    xgb_val_pred
)

xgb_test_results = evaluate_classifier(
    "XGBoost",
    y_test,
    xgb_test_pred
)

print("Validation:")
print(xgb_val_results)

print("\nTest:")
print(xgb_test_results)

Validation:
{'Model': 'XGBoost', 'Accuracy': 0.759493670886076, 'Macro_F1': 0.6723810399003719, 'Weighted_F1': 0.7650985500054435, 'Macro_Precision': 0.8054718244582629, 'Macro_Recall': 0.6128650879943406}

Test:
{'Model': 'XGBoost', 'Accuracy': 0.7383966244725738, 'Macro_F1': 0.6864257661907357, 'Weighted_F1': 0.7485547073750617, 'Macro_Precision': 0.8350445097901857, 'Macro_Recall': 0.6188508375485102}


In [8]:
# ============================================================
# FINAL CLASSIFIER COMPARISON
# ============================================================

results = pd.DataFrame([
    lr_test_results,
    linear_test_results,
    mlp_test_results,
    svm_test_results,
    xgb_test_results
])

results = results.sort_values(
    "Macro_F1",
    ascending=False
).reset_index(drop=True)

print("\n==============================================")
print("GENEFORMER CLASSIFIER COMPARISON")
print("==============================================")

print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


GENEFORMER CLASSIFIER COMPARISON
              Model  Accuracy  Macro_F1  Weighted_F1  Macro_Precision  Macro_Recall
         Linear SVM    0.9204    0.8963       0.9194           0.9021        0.8931
Logistic Regression    0.9216    0.8923       0.9198           0.8995        0.8891
                MLP    0.9114    0.8809       0.9087           0.8853        0.8813
 Linear Neural Head    0.8957    0.8559       0.8903           0.8738        0.8596
            XGBoost    0.7384    0.6864       0.7486           0.8350        0.6189


In [29]:
# ============================================================
# SAVE COMPLETE CLASSIFIER STUDY RESULTS
# Geneformer TCGA Pan-Cancer Classification
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)


# ============================================================
# 0. OUTPUT DIRECTORY
# ============================================================

RESULTS_DIR = r"F:\GeneformerProject\results\classifier_study"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("=" * 80)
print("CLASSIFIER STUDY — SAVING RESULTS")
print("=" * 80)
print("\nResults directory:")
print(RESULTS_DIR)


# ============================================================
# 1. CANCER NAMES
# ============================================================

cancer_names = [
    "acute myeloid leukemia",
    "adrenocortical cancer",
    "bladder urothelial carcinoma",
    "brain lower grade glioma",
    "breast invasive carcinoma",
    "cervical & endocervical cancer",
    "cholangiocarcinoma",
    "colon adenocarcinoma",
    "diffuse large B-cell lymphoma",
    "esophageal carcinoma",
    "glioblastoma multiforme",
    "head & neck squamous cell carcinoma",
    "kidney chromophobe",
    "kidney clear cell carcinoma",
    "kidney papillary cell carcinoma",
    "liver hepatocellular carcinoma",
    "lung adenocarcinoma",
    "lung squamous cell carcinoma",
    "mesothelioma",
    "ovarian serous cystadenocarcinoma",
    "pancreatic adenocarcinoma",
    "pheochromocytoma & paraganglioma",
    "prostate adenocarcinoma",
    "rectum adenocarcinoma",
    "sarcoma",
    "skin cutaneous melanoma",
    "stomach adenocarcinoma",
    "testicular germ cell tumor",
    "thymoma",
    "thyroid carcinoma",
    "uterine carcinosarcoma",
    "uterine corpus endometrioid carcinoma",
    "uveal melanoma"
]

NUM_CLASSES = len(cancer_names)

print("\nNumber of cancer classes:", NUM_CLASSES)


# ============================================================
# 2. BASIC DATA CHECKS
# ============================================================

print("\n" + "=" * 80)
print("DATA CHECKS")
print("=" * 80)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

assert X_train.shape[1] == 768
assert X_val.shape[1] == 768
assert X_test.shape[1] == 768

assert len(np.unique(y_train)) == NUM_CLASSES
assert len(np.unique(y_val)) == NUM_CLASSES
assert len(np.unique(y_test)) == NUM_CLASSES

assert len(cancer_names) == NUM_CLASSES

print("Data checks passed.")


# ============================================================
# 3. RECREATE TEST PREDICTIONS
# ============================================================

print("\n" + "=" * 80)
print("CREATING TEST PREDICTIONS")
print("=" * 80)


# ------------------------------------------------------------
# Logistic Regression
# ------------------------------------------------------------

lr_test_pred = lr.predict(X_test)


# ------------------------------------------------------------
# Linear SVM
# ------------------------------------------------------------

svm_test_pred = svm.predict(X_test)


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

xgb_test_pred = xgb.predict(X_test)


# ------------------------------------------------------------
# Linear Neural Head
# ------------------------------------------------------------

linear_head.eval()

with torch.no_grad():

    linear_test_logits = linear_head(
        X_test_tensor.to(device)
    )

    linear_test_pred = torch.argmax(
        linear_test_logits,
        dim=1
    ).cpu().numpy()


# ------------------------------------------------------------
# MLP
# ------------------------------------------------------------

mlp.eval()

with torch.no_grad():

    mlp_test_logits = mlp(
        X_test_tensor.to(device)
    )

    mlp_test_pred = torch.argmax(
        mlp_test_logits,
        dim=1
    ).cpu().numpy()


# ============================================================
# CHECK PREDICTIONS
# ============================================================

models_predictions = {

    "Logistic Regression":
        lr_test_pred,

    "Linear Neural Head":
        linear_test_pred,

    "MLP":
        mlp_test_pred,

    "Linear SVM":
        svm_test_pred,

    "XGBoost":
        xgb_test_pred
}


print("\nPrediction shapes:")

for model_name, predictions in models_predictions.items():

    print(
        f"{model_name:25s}: "
        f"{predictions.shape}"
    )

    assert predictions.shape == y_test.shape

print("\nPrediction checks passed.")


# ============================================================
# 4. VALIDATION RESULTS
# ============================================================

print("\n" + "=" * 80)
print("VALIDATION RESULTS")
print("=" * 80)


# ------------------------------------------------------------
# Recalculate validation predictions for neural models
# ------------------------------------------------------------

linear_head.eval()

with torch.no_grad():

    linear_val_logits = linear_head(
        X_val_tensor.to(device)
    )

    linear_val_pred = torch.argmax(
        linear_val_logits,
        dim=1
    ).cpu().numpy()


mlp.eval()

with torch.no_grad():

    mlp_val_logits = mlp(
        X_val_tensor.to(device)
    )

    mlp_val_pred = torch.argmax(
        mlp_val_logits,
        dim=1
    ).cpu().numpy()


# ------------------------------------------------------------
# Calculate validation metrics
# ------------------------------------------------------------

linear_val_results = evaluate_classifier(
    "Linear Neural Head",
    y_val,
    linear_val_pred
)

mlp_val_results = evaluate_classifier(
    "MLP",
    y_val,
    mlp_val_pred
)


# Existing models
# lr_val_results, svm_val_results and xgb_val_results
# should already exist from your training sections.

all_validation_results = pd.DataFrame([

    lr_val_results,

    linear_val_results,

    mlp_val_results,

    svm_val_results,

    xgb_val_results
])


all_validation_results = (
    all_validation_results
    .sort_values("Macro_F1", ascending=False)
    .reset_index(drop=True)
)


all_validation_results.to_csv(
    os.path.join(
        RESULTS_DIR,
        "validation_results.csv"
    ),
    index=False
)


print(
    all_validation_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 5. TEST RESULTS
# ============================================================

print("\n" + "=" * 80)
print("TEST RESULTS")
print("=" * 80)


all_test_results = pd.DataFrame([

    evaluate_classifier(
        "Logistic Regression",
        y_test,
        lr_test_pred
    ),

    evaluate_classifier(
        "Linear Neural Head",
        y_test,
        linear_test_pred
    ),

    evaluate_classifier(
        "MLP",
        y_test,
        mlp_test_pred
    ),

    evaluate_classifier(
        "Linear SVM",
        y_test,
        svm_test_pred
    ),

    evaluate_classifier(
        "XGBoost",
        y_test,
        xgb_test_pred
    )
])


all_test_results = (
    all_test_results
    .sort_values("Macro_F1", ascending=False)
    .reset_index(drop=True)
)


all_test_results.to_csv(
    os.path.join(
        RESULTS_DIR,
        "test_results.csv"
    ),
    index=False
)


print(
    all_test_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 6. TEST SCORES / PROBABILITIES FOR ROC-AUC
# ============================================================

print("\n" + "=" * 80)
print("CALCULATING ROC-AUC SCORES")
print("=" * 80)


# ------------------------------------------------------------
# Logistic Regression
# ------------------------------------------------------------

lr_scores = lr.predict_proba(X_test)


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

xgb_scores = xgb.predict_proba(X_test)


# ------------------------------------------------------------
# Linear Neural Head
# ------------------------------------------------------------

linear_head.eval()

with torch.no_grad():

    linear_test_logits = linear_head(
        X_test_tensor.to(device)
    )

    linear_scores = torch.softmax(
        linear_test_logits,
        dim=1
    ).cpu().numpy()


# ------------------------------------------------------------
# MLP
# ------------------------------------------------------------

mlp.eval()

with torch.no_grad():

    mlp_test_logits = mlp(
        X_test_tensor.to(device)
    )

    mlp_scores = torch.softmax(
        mlp_test_logits,
        dim=1
    ).cpu().numpy()


# ------------------------------------------------------------
# Linear SVM
# ------------------------------------------------------------

# SVM does not necessarily provide probabilities.
# decision_function provides continuous class scores,
# which are appropriate for ROC-AUC.

svm_scores = svm.decision_function(X_test)


# ============================================================
# CHECK SCORE MATRICES
# ============================================================

score_models = {

    "Logistic Regression":
        lr_scores,

    "Linear Neural Head":
        linear_scores,

    "MLP":
        mlp_scores,

    "Linear SVM":
        svm_scores,

    "XGBoost":
        xgb_scores
}


print("\nScore matrix shapes:")

for model_name, scores in score_models.items():

    print(
        f"{model_name:25s}: "
        f"{scores.shape}"
    )

    assert scores.shape == (
        len(y_test),
        NUM_CLASSES
    )

print("\nScore checks passed.")


# ============================================================
# 7. MULTICLASS ROC-AUC
# ============================================================

auc_results = []


for model_name, scores in score_models.items():

    macro_auc = roc_auc_score(
        y_test,
        scores,
        multi_class="ovr",
        average="macro"
    )

    weighted_auc = roc_auc_score(
        y_test,
        scores,
        multi_class="ovr",
        average="weighted"
    )

    auc_results.append({

        "Model":
            model_name,

        "Macro_AUC":
            macro_auc,

        "Weighted_AUC":
            weighted_auc
    })


auc_df = pd.DataFrame(
    auc_results
)


auc_df = auc_df.sort_values(
    "Macro_AUC",
    ascending=False
).reset_index(drop=True)


auc_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "roc_auc_results.csv"
    ),
    index=False
)


print("\nROC-AUC results:")

print(
    auc_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 8. PER-CANCER AUC
# ============================================================

print("\n" + "=" * 80)
print("PER-CANCER ROC-AUC")
print("=" * 80)


all_per_cancer_auc = []


for model_name, scores in score_models.items():

    class_auc = roc_auc_score(
        y_test,
        scores,
        multi_class="ovr",
        average=None
    )

    assert len(class_auc) == NUM_CLASSES

    for class_id, cancer in enumerate(cancer_names):

        all_per_cancer_auc.append({

            "Model":
                model_name,

            "Cancer_Type":
                cancer,

            "Class_ID":
                class_id,

            "AUC":
                class_auc[class_id]
        })


per_cancer_auc_df = pd.DataFrame(
    all_per_cancer_auc
)


per_cancer_auc_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "per_cancer_auc_all_models.csv"
    ),
    index=False
)


print(
    per_cancer_auc_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 9. SAVE TEST PREDICTIONS
# ============================================================

print("\n" + "=" * 80)
print("SAVING TEST PREDICTIONS")
print("=" * 80)


predictions_df = pd.DataFrame({

    "True_Label":
        y_test,

    "True_Cancer":
        [
            cancer_names[i]
            for i in y_test
        ],

    "Logistic_Regression":
        lr_test_pred,

    "Linear_Neural_Head":
        linear_test_pred,

    "MLP":
        mlp_test_pred,

    "Linear_SVM":
        svm_test_pred,

    "XGBoost":
        xgb_test_pred
})


# ------------------------------------------------------------
# Predicted cancer names
# ------------------------------------------------------------

predictions_df["LR_Cancer"] = [
    cancer_names[i]
    for i in lr_test_pred
]

predictions_df["Linear_Head_Cancer"] = [
    cancer_names[i]
    for i in linear_test_pred
]

predictions_df["MLP_Cancer"] = [
    cancer_names[i]
    for i in mlp_test_pred
]

predictions_df["SVM_Cancer"] = [
    cancer_names[i]
    for i in svm_test_pred
]

predictions_df["XGBoost_Cancer"] = [
    cancer_names[i]
    for i in xgb_test_pred
]


predictions_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "test_predictions_all_models.csv"
    ),
    index=False
)


print(
    "Saved: test_predictions_all_models.csv"
)


# ============================================================
# 10. CONFUSION MATRICES
# ============================================================

print("\n" + "=" * 80)
print("SAVING CONFUSION MATRICES")
print("=" * 80)


for model_name, predictions in models_predictions.items():

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=np.arange(NUM_CLASSES)
    )


    cm_df = pd.DataFrame(
        cm,
        index=cancer_names,
        columns=cancer_names
    )


    safe_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


    cm_df.to_csv(
        os.path.join(
            RESULTS_DIR,
            f"confusion_matrix_{safe_name}.csv"
        )
    )


    print(
        f"Saved confusion matrix: {model_name}"
    )


# ============================================================
# 11. PER-CANCER CLASSIFICATION PERFORMANCE
# ============================================================

print("\n" + "=" * 80)
print("SAVING PER-CANCER CLASSIFICATION PERFORMANCE")
print("=" * 80)


all_per_cancer = []


for model_name, predictions in models_predictions.items():

    report = classification_report(

        y_test,

        predictions,

        labels=np.arange(NUM_CLASSES),

        target_names=cancer_names,

        output_dict=True,

        zero_division=0
    )


    for class_id, cancer in enumerate(cancer_names):

        metrics = report[cancer]


        all_per_cancer.append({

            "Model":
                model_name,

            "Cancer_Type":
                cancer,

            "Class_ID":
                class_id,

            "Precision":
                metrics["precision"],

            "Recall":
                metrics["recall"],

            "F1":
                metrics["f1-score"],

            "Support":
                metrics["support"]
        })


per_cancer_df = pd.DataFrame(
    all_per_cancer
)


per_cancer_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "per_cancer_performance_all_models.csv"
    ),
    index=False
)


print(
    "Saved: per_cancer_performance_all_models.csv"
)


# ============================================================
# 12. FINAL MASTER RESULTS TABLE
# ============================================================

print("\n" + "=" * 80)
print("FINAL CLASSIFIER COMPARISON")
print("=" * 80)


final_results = all_test_results.merge(
    auc_df,
    on="Model",
    how="left"
)


final_results = (
    final_results
    .sort_values("Macro_F1", ascending=False)
    .reset_index(drop=True)
)


final_results.to_csv(
    os.path.join(
        RESULTS_DIR,
        "FINAL_CLASSIFIER_COMPARISON.csv"
    ),
    index=False
)


print(
    final_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 13. EXPERIMENT CONFIGURATION
# ============================================================

config = {

    "Project":
        "TCGA Pan-Cancer Classification using Geneformer",

    "Embedding_Model":
        "Geneformer-V2-104M",

    "Embedding_Type":
        "Frozen CLS embedding",

    "Embedding_Dimension":
        768,

    "Number_of_Cancer_Classes":
        NUM_CLASSES,

    "Train_Samples":
        len(y_train),

    "Validation_Samples":
        len(y_val),

    "Test_Samples":
        len(y_test),

    "Train_Validation_Test_Split":
        "70/15/15 stratified by Cancer_Type",

    "Random_State":
        42,

    "Sequence_Length":
        4096,

    "Top_Genes":
        4094,

    "Tokenization":
        "Custom bulk-TCGA rank-based Geneformer tokenization",

    "Expression_Preprocessing":
        "Train-fitted mean imputation; no additional log/scaling",

    "Classifier_1":
        "Logistic Regression",

    "Classifier_2":
        "Linear Neural Head",

    "Classifier_3":
        "MLP 768-256-33",

    "Classifier_4":
        "Linear SVM",

    "Classifier_5":
        "XGBoost",

    "AUC_Method":
        "One-vs-Rest multiclass ROC-AUC",

    "Model_Selection":
        "Validation Macro-F1",

    "Test_Set_Usage":
        "Final evaluation only"
}


config_df = pd.DataFrame(
    list(config.items()),
    columns=[
        "Parameter",
        "Value"
    ]
)


config_df.to_csv(
    os.path.join(
        RESULTS_DIR,
        "experiment_configuration.csv"
    ),
    index=False
)


print(
    "Saved: experiment_configuration.csv"
)


# ============================================================
# 14. FINAL FILE LIST
# ============================================================

print("\n" + "=" * 80)
print("ALL RESULTS SAVED")
print("=" * 80)

saved_files = sorted(
    os.listdir(RESULTS_DIR)
)

for filename in saved_files:

    print(
        " -",
        filename
    )

print("\nCompleted successfully.")

Results will be saved to:
F:\GeneformerProject\results\classifier_study

Validation results:
              Model  Accuracy  Macro_F1  Weighted_F1  Macro_Precision  Macro_Recall
         Linear SVM  0.924653  0.886026     0.922793         0.895489      0.884825
Logistic Regression  0.923448  0.883695     0.920506         0.888141      0.884567
                MLP  0.915009  0.875974     0.912265         0.880185      0.879342
 Linear Neural Head  0.908981  0.870613     0.907025         0.879182      0.875068
            XGBoost  0.878240  0.826356     0.870458         0.882383      0.803864

Test results:
              Model  Accuracy  Macro_F1  Weighted_F1  Macro_Precision  Macro_Recall
         Linear SVM  0.920434  0.896296     0.919383         0.902104      0.893106
Logistic Regression  0.921640  0.892271     0.919816         0.899548      0.889070
 Linear Neural Head  0.903556  0.871049     0.902317         0.884923      0.871871
                MLP  0.910790  0.865470     0.907087

In [36]:
# ============================================================
# FINAL MASTER RESULTS TABLE
# ============================================================

final_results = all_test_results.merge(
    auc_df,
    on="Model",
    how="left"
)

final_results = final_results.sort_values(
    "Macro_F1",
    ascending=False
).reset_index(drop=True)


final_results.to_csv(
    os.path.join(
        RESULTS_DIR,
        "FINAL_CLASSIFIER_COMPARISON.csv"
    ),
    index=False
)


print("\n")
print("=" * 80)
print("FINAL CLASSIFIER COMPARISON")
print("=" * 80)

print(
    final_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)



FINAL CLASSIFIER COMPARISON
              Model  Accuracy  Macro_F1  Weighted_F1  Macro_Precision  Macro_Recall  Macro_AUC  Weighted_AUC
         Linear SVM    0.9204    0.8963       0.9194           0.9021        0.8931     0.9952        0.9950
Logistic Regression    0.9216    0.8923       0.9198           0.8995        0.8891     0.9980        0.9979
 Linear Neural Head    0.9036    0.8710       0.9023           0.8849        0.8719     0.9970        0.9970
                MLP    0.9108    0.8655       0.9071           0.8649        0.8729     0.9972        0.9975
            XGBoost    0.8626    0.8194       0.8553           0.8560        0.8043     0.9932        0.9937
